# Loan Approval Prediction (Part B2)

Assignment Five — Classification: Theory and Practice

This notebook trains and compares three classifiers on the cleaned loan dataset: Logistic Regression, Random Forest, and K-Nearest Neighbors (my chosen third algorithm).

## Step 1 — Load Dataset

In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("clean_loan_dataset.csv")
print("Shape:", df.shape)
df.head()


Shape: (100, 9)


## Step 2 — Prepare Features & Target

In [2]:
X = df.drop(columns=['Approved'])
y = df['Approved']

print("Features:", list(X.columns))
print("Target: Approved")
print(y.value_counts())


Features: ['Income', 'CreditScore', 'EmploymentYears', 'LoanAmount', 'HasCollateral', 'PreviousDefaults', 'DebtToIncome', 'IncomePerYearEmployed']
Target: Approved
Approved
1    66
0    34
Name: count, dtype: int64


## Step 3 — Split Data

80% train / 20% test, stratified on `y`, with `random_state=42` for reproducibility.

In [3]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)
print("Train class balance:")
print(y_train.value_counts(normalize=True))


Train shape: (80, 8)
Test shape: (20, 8)
Train class balance:
Approved
1    0.6625
0    0.3375
Name: proportion, dtype: float64


## Step 4 — Train Models (Reproduce Lesson)

Logistic Regression and Random Forest, matching the Lesson 5 coding session.

In [4]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

log_reg = LogisticRegression(max_iter=1000, random_state=42)
log_reg.fit(X_train, y_train)

rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)

print("Logistic Regression and Random Forest trained.")


Logistic Regression and Random Forest trained.


## Step 5 — Research & Train a Third Classifier

**Chosen algorithm: K-Nearest Neighbors (KNN)**

KNN classifies a new applicant by looking at the `k` most similar applicants (by distance in feature space) in the training set and taking a majority vote of their labels. It fits loan approval prediction reasonably well because applicants with similar income, credit score, employment history, and debt ratios often end up with similar approval outcomes — the "nearest neighbors" idea maps naturally onto "similar applicants get similar decisions."

- **Advantage:** simple, no training phase (it just stores the data), and it can capture non-linear decision boundaries that Logistic Regression would miss.
- **Limitation:** it is sensitive to feature scaling (which is why Step 10 of the preprocessing notebook mattered) and can get slow/less reliable as the dataset grows large, since every prediction requires comparing against many stored points.

**Source:** scikit-learn documentation, *Nearest Neighbors Classification* — https://scikit-learn.org/stable/modules/neighbors.html#classification

In [5]:
from sklearn.neighbors import KNeighborsClassifier

knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train, y_train)

print("KNN trained with n_neighbors=5.")


KNN trained with n_neighbors=5.


## Step 6 — Evaluate Performance

In [6]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

def evaluate(name, model, X_test, y_test):
    preds = model.predict(X_test)
    acc = accuracy_score(y_test, preds)
    prec = precision_score(y_test, preds, pos_label=1, zero_division=0)
    rec = recall_score(y_test, preds, pos_label=1, zero_division=0)
    f1 = f1_score(y_test, preds, pos_label=1, zero_division=0)

    print(f"{name} Performance:")
    print(f"  Accuracy : {acc:.3f}")
    print(f"  Precision: {prec:.3f}  (positive = Approved=1)")
    print(f"  Recall   : {rec:.3f}  (positive = Approved=1)")
    print(f"  F1-Score : {f1:.3f}  (positive = Approved=1)")
    print()
    return preds

lr_preds = evaluate("Logistic Regression", log_reg, X_test, y_test)
rf_preds = evaluate("Random Forest", rf, X_test, y_test)
knn_preds = evaluate("K-Nearest Neighbors", knn, X_test, y_test)


Logistic Regression Performance:
  Accuracy : 0.700
  Precision: 0.733  (positive = Approved=1)
  Recall   : 0.846  (positive = Approved=1)
  F1-Score : 0.786  (positive = Approved=1)

Random Forest Performance:
  Accuracy : 0.650
  Precision: 0.714  (positive = Approved=1)
  Recall   : 0.769  (positive = Approved=1)
  F1-Score : 0.741  (positive = Approved=1)

K-Nearest Neighbors Performance:
  Accuracy : 0.650
  Precision: 0.688  (positive = Approved=1)
  Recall   : 0.846  (positive = Approved=1)
  F1-Score : 0.759  (positive = Approved=1)



In [7]:
print("Logistic Regression Confusion Matrix:")
print(confusion_matrix(y_test, lr_preds))
print()
print("Random Forest Confusion Matrix:")
print(confusion_matrix(y_test, rf_preds))
print()
print("K-Nearest Neighbors Confusion Matrix:")
print(confusion_matrix(y_test, knn_preds))


Logistic Regression Confusion Matrix:
[[ 3  4]
 [ 2 11]]

Random Forest Confusion Matrix:
[[ 3  4]
 [ 3 10]]

K-Nearest Neighbors Confusion Matrix:
[[ 2  5]
 [ 2 11]]


## Step 7 — Sanity Check

Compare all three models' predictions against the actual label for one test-set applicant.

In [8]:
i = 0
row = X_test.iloc[[i]]
actual = y_test.iloc[i]

print("Test row features:")
print(row)
print()
print("Actual Approved:           ", actual)
print("Logistic Regression predicts:", log_reg.predict(row)[0])
print("Random Forest predicts:      ", rf.predict(row)[0])
print("KNN predicts:                ", knn.predict(row)[0])


Test row features:
      Income  CreditScore  ...  DebtToIncome  IncomePerYearEmployed
70  0.971475    -0.517799  ...      0.844548               0.178435

[1 rows x 8 columns]

Actual Approved:            1
Logistic Regression predicts: 0
Random Forest predicts:       0
KNN predicts:                 0
